# 第 10 章 大语言模型与智能体（下）：微调与智能体

上一节我们从零训了一个 nanoGPT，它能生成像莎士比亚风格的文本。但是预训练完的模型还不能**听话**——你跟它说"请用一句话总结《罗密欧与朱丽叶》"，它可能继续 babble 莎翁式独白。

让模型听话需要两步：

1. **监督微调**（Supervised Fine-Tuning，SFT）：在"指令-回答"数据上继续训练；
2. **人类反馈对齐**（RLHF / DPO）：用人类偏好数据调整模型行为。

本节用极简的玩具数据演示 LoRA-SFT 和 DPO，最后实现一个 **ReAct 风格**的智能体雏形。


## 1. 参数高效微调：LoRA

直接微调大模型的所有参数（**全量微调**）成本很高——比如一个 7B 的 Llama，全量微调需要 ~80 GB 显存。

**LoRA**（Low-Rank Adaptation，Hu et al., 2022）的思路：冻结预训练权重 $\boldsymbol{W}_0$，只学习一个低秩**增量** $\Delta \boldsymbol{W} = \boldsymbol{B} \boldsymbol{A}$，其中 $\boldsymbol{A} \in \mathbb{R}^{r \times d}$、$\boldsymbol{B} \in \mathbb{R}^{d \times r}$，秩 $r \ll d$。前向变成

$$ \boldsymbol{h} = \boldsymbol{W}_0 \boldsymbol{x} + \boldsymbol{B} \boldsymbol{A} \boldsymbol{x}. $$

参数量从 $d^2$ 降到 $2dr$，通常缩减 100-1000 倍。下面用一个最小实现把它套到 `nn.Linear` 上。


In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class LoRALinear(nn.Module):
    """包装一个 nn.Linear，加上 LoRA 增量 (rank-r) 适配器。
    原 Linear 的 weight 被冻结，只训练 A 和 B。
    """

    def __init__(self, base: nn.Linear, r=4, alpha=16):
        super().__init__()
        self.base = base
        for p in self.base.parameters():
            p.requires_grad = False                  # 冻结底层 Linear

        self.r = r
        self.scaling = alpha / r                     # LoRA 论文推荐的缩放
        in_f, out_f = base.in_features, base.out_features
        self.A = nn.Parameter(torch.zeros(r, in_f))
        self.B = nn.Parameter(torch.zeros(out_f, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))  # 论文里 A 用 He 初始化
        # B 保持全 0：训练开始时 BA = 0，等价于原模型

    def forward(self, x):
        # base(x) + (x @ A^T @ B^T) * scaling
        return self.base(x) + (x @ self.A.t() @ self.B.t()) * self.scaling


# 演示：包装一个 Linear，统计可训练参数
base_lin = nn.Linear(in_features=128, out_features=128, bias=False)
lora_lin = LoRALinear(base_lin, r=4, alpha=16)
full_params = sum(p.numel() for p in base_lin.parameters())
train_params = sum(p.numel() for p in lora_lin.parameters() if p.requires_grad)
print(f'全量参数: {full_params:,}')
print(f'LoRA 可训练参数: {train_params:,}  (r=4)')
print(f'压缩比: {full_params / train_params:.1f}x')


## 2. SFT 玩具示例：让模型学会大小写转换

为了让 SFT 的概念可视化，我们设计一个极简任务：

> 给定一段小写文本，模型要输出对应的大写。比如 `'hello world'` → `'HELLO WORLD'`。

这是一个完全确定性的任务，但对一个没见过这种模式的小模型来说仍然需要学习——它得理解"指令"是什么，输出什么。我们用纯字符级 GPT，喂它几百对 `(小写, 大写)` 训练样本，看 SFT 后能不能泛化到新句子。

注意：在真实 LLM 场景里，SFT 数据是 *(指令, 回答)* 对（如 Alpaca 数据集），模型学会"被指令"。这里玩具任务原理一致，只是简化到字符串级别。


In [ ]:
import random


# ============== 玩具 SFT 数据集 ==============
def make_sft_pair(seed):
    rng = random.Random(seed)
    words = ['hello', 'world', 'pytorch', 'transformer', 'language', 'model',
             'small', 'cat', 'dog', 'machine', 'learn', 'deep', 'neural', 'net']
    n_words = rng.randint(2, 4)
    src = ' '.join(rng.choice(words) for _ in range(n_words))
    tgt = src.upper()
    # 格式化为 '<lower>=<upper>' 这种 prompt-response 形式
    return f'{src}=', tgt + '\n'


# 字符词表 = a-z + A-Z + space + '=' + '\n'
vocab = list('abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ =\n')
stoi = {c: i for i, c in enumerate(vocab)}
itos = {i: c for i, c in enumerate(vocab)}
V = len(vocab)


def encode(s): return torch.tensor([stoi[c] for c in s], dtype=torch.long)
def decode(ids): return ''.join(itos[int(i)] for i in ids)


# 生成 200 个训练样本和 50 个测试样本
train_pairs = [make_sft_pair(i) for i in range(200)]
test_pairs  = [make_sft_pair(1000 + i) for i in range(50)]
print('sample pairs:')
for src, tgt in train_pairs[:3]:
    print(f'  prompt={src!r}  target={tgt!r}')


In [ ]:
# ============== 一个极小的 GPT（结构简化版） ==============
class TinyGPT(nn.Module):
    """很简化的 GPT：1 层 attention + 1 层 FFN，方便演示 SFT。"""

    def __init__(self, vocab_size, block_size, n_embd=64, n_head=4):
        super().__init__()
        self.block_size = block_size
        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(block_size, n_embd)
        # 一个 Transformer 层
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = nn.MultiheadAttention(n_embd, n_head, batch_first=True)
        self.ln2 = nn.LayerNorm(n_embd)
        self.ffn = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd), nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
        )
        self.ln_f = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size, bias=False)
        # 因果掩码
        mask = torch.triu(torch.ones(block_size, block_size), diagonal=1).bool()
        self.register_buffer('mask', mask)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok = self.tok_emb(idx)
        pos = self.pos_emb(torch.arange(T, device=idx.device))
        x = tok + pos
        # attention
        h = self.ln1(x)
        attn_mask = self.mask[:T, :T]
        h, _ = self.attn(h, h, h, attn_mask=attn_mask, need_weights=False)
        x = x + h
        # ffn
        x = x + self.ffn(self.ln2(x))
        logits = self.head(self.ln_f(x))
        loss = None
        if targets is not None:
            # ignore_index=-100：在 prompt 部分不算 loss，只在 response 上算
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)),
                                   targets.view(-1), ignore_index=-100)
        return logits, loss


torch.manual_seed(0)
block_size = 64
model = TinyGPT(vocab_size=V, block_size=block_size, n_embd=64, n_head=4)
print('TinyGPT params:', sum(p.numel() for p in model.parameters()))


### 2.1 SFT 训练循环

关键技巧：**只在 response 部分计算 loss**。Prompt 部分的 token 是"输入条件"，让模型学习去预测它没有意义。具体做法是把 targets 里 prompt 部分的位置填 `-100`，PyTorch 的 `cross_entropy` 会自动忽略 `-100` 的位置。


In [ ]:
def build_sft_batch(pairs, block_size):
    """
    把 (prompt, target) 拼成单串：prompt + target。
    targets：prompt 段填 -100（不算 loss），target 段填 target token id。
    """
    xs, ys = [], []
    for prompt, target in pairs:
        full = prompt + target
        if len(full) > block_size:
            full = full[:block_size]
        ids = encode(full)
        x = ids[:-1]
        y = ids[1:].clone()
        # mask 掉 prompt 部分（prompt 之前的部分 + prompt 本身）的 loss
        # x[t] 预测 y[t]；y[t] 对应原文 ids[t+1] 的字符
        # 我们想从 ids[len(prompt)] 开始算 loss
        for t in range(len(prompt) - 1):           # -1 因为 x 是 ids[:-1]
            y[t] = -100
        # padding 到 block_size
        pad = block_size - len(x)
        if pad > 0:
            x = torch.cat([x, torch.zeros(pad, dtype=torch.long)])
            y = torch.cat([y, torch.full((pad,), -100, dtype=torch.long)])
        xs.append(x); ys.append(y)
    return torch.stack(xs), torch.stack(ys)


optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3)
batch_size = 16

for step in range(500):
    batch = random.sample(train_pairs, batch_size)
    X, Y = build_sft_batch(batch, block_size)
    _, loss = model(X, Y)
    optimizer.zero_grad(); loss.backward(); optimizer.step()
    if step % 100 == 0 or step == 499:
        print(f'step {step:3d}  loss {loss.item():.4f}')


# ============== 看效果：在没见过的输入上推断 ==============
@torch.no_grad()
def generate_until(model, prompt, max_new=30, stop_char='\n'):
    model.eval()
    ids = encode(prompt).unsqueeze(0)
    for _ in range(max_new):
        if ids.size(1) > model.block_size:
            ids = ids[:, -model.block_size:]
        logits, _ = model(ids)
        next_id = logits[0, -1].argmax(dim=-1, keepdim=True).unsqueeze(0)
        ids = torch.cat([ids, next_id], dim=1)
        if itos[int(next_id.item())] == stop_char:
            break
    return decode(ids[0].tolist())


for prompt, target in test_pairs[:5]:
    out = generate_until(model, prompt)
    print(f'prompt: {prompt!r:40s}  pred: {out!r:60s}  target: {target!r}')


**典型观察**：500 步左右，模型基本能学会"把 prompt 里的小写转大写"这个模式——即使是没见过的单词组合也能正确输出。这就是 SFT 让模型"学会指令"的本质：在(prompt, response)数据上让模型预测 response。


## 3. DPO：直接偏好优化

SFT 让模型会"听话"，但还不能让模型"喜欢"某种风格胜过另一种。比如同样是回答"今天天气如何"，模型可能说"晴朗"，也可能说"我无法预测天气"——SFT 数据没法告诉它哪个更好。

**RLHF**（Reinforcement Learning from Human Feedback）的做法：
1. 收集人类偏好数据 $(x, y_w, y_l)$——同一个 prompt $x$，标注员认为回答 $y_w$（win）比 $y_l$（lose）更好；
2. 用偏好数据训练奖励模型 $r_\phi(x, y)$；
3. 用 PPO 等 RL 算法优化模型最大化奖励。

**DPO**（Direct Preference Optimization，Rafailov et al., 2023）观察到一个数学事实：上述三步可以**合并成一个单一的对比损失**，不需要单独训奖励模型，也不需要 RL。DPO 损失是

$$ \mathcal{L}_\mathrm{DPO} = -\log \sigma\Big(\beta \log\frac{\pi_\theta(y_w \mid x)}{\pi_\mathrm{ref}(y_w \mid x)} - \beta \log\frac{\pi_\theta(y_l \mid x)}{\pi_\mathrm{ref}(y_l \mid x)}\Big), $$

其中 $\pi_\theta$ 是要训练的策略（语言模型），$\pi_\mathrm{ref}$ 是冻结的参考模型（通常是 SFT 后的模型），$\beta$ 控制偏离参考模型的程度。

直觉：让 $\pi_\theta$ 把更多概率质量放在 $y_w$ 上、更少放在 $y_l$ 上，但同时不要离 $\pi_\mathrm{ref}$ 太远。


In [ ]:
def logprob_of_sequence(model, prompt, response):
    """算 log p(response | prompt) = sum_t log p(response[t] | prompt + response[<t])."""
    full = prompt + response
    ids = encode(full).unsqueeze(0)
    # 用模型预测 next token
    logits, _ = model(ids)
    # 我们要的是 prompt 后面每一位的 log prob
    # ids[:, t] 预测 ids[:, t+1]，所以 logits[:, t] 对应 ids[:, t+1]
    log_probs = F.log_softmax(logits[0], dim=-1)      # [T, V]
    # 找到 response 在 full 里的位置
    p_len = len(prompt)
    target_ids = ids[0, p_len:]                       # response 的 token ids
    # logits[p_len-1] 预测 ids[p_len]（即 response 第一个字符）
    chosen_log_probs = log_probs[p_len-1:p_len-1+len(target_ids), :]
    return chosen_log_probs.gather(1, target_ids.unsqueeze(1)).sum().item()


def dpo_loss(policy, ref, prompt, y_win, y_lose, beta=0.1):
    """DPO 损失：让 policy 偏好 y_win 而非 y_lose。"""
    # log pi_policy(y | x)
    lp_w = logprob_of_sequence(policy, prompt, y_win)
    lp_l = logprob_of_sequence(policy, prompt, y_lose)
    # log pi_ref(y | x)
    with torch.no_grad():
        lr_w = logprob_of_sequence(ref, prompt, y_win)
        lr_l = logprob_of_sequence(ref, prompt, y_lose)
    # logits = beta * [(lp_w - lr_w) - (lp_l - lr_l)]
    logits = beta * ((lp_w - lr_w) - (lp_l - lr_l))
    loss = -F.logsigmoid(torch.tensor(logits))
    return loss


# 演示：在一个简单偏好对上调用 DPO（仅展示前向，不真训）
import copy

ref_model = copy.deepcopy(model)
for p in ref_model.parameters():
    p.requires_grad = False
ref_model.eval()

prompt = 'pytorch='
y_win  = 'PYTORCH\n'                                 # 完整的大写答案（偏好）
y_lose = 'pytorch\n'                                 # 错的（拒绝）
loss = dpo_loss(model, ref_model, prompt, y_win, y_lose, beta=0.1)
print(f'DPO loss for win=PYTORCH vs lose=pytorch: {loss.item():.4f}')


**关键认识**：

- DPO 把 RLHF 三步并成一步，实现简洁、训练稳定；
- 它需要冻结的参考模型 `ref` ——通常是 SFT 后的模型副本；
- 实际使用时 `huggingface/trl` 库提供了完整的 `DPOTrainer`，可以一行调用。

> 上面只是 DPO loss 的最小演示。完整 DPO 训练需要批量、KL 估计、参考模型管理等工程细节，建议生产环境用现成的库（`trl.DPOTrainer`）。


## 4. ReAct 智能体雏形

把语言模型变成**智能体**（Agent）的核心思路：让 LM 不仅输出文本，还**和外部世界交互**——调用搜索引擎、运行代码、查询数据库等。

**ReAct**（Yao et al., 2022）框架交替进行三种动作：

- **Thought**：模型自言自语推理；
- **Action**：调用一个工具（比如 `Search("...")`、`Calculator("...")`）；
- **Observation**：工具返回的结果，输入回模型。

直到模型产生最终的 `Answer` 才停止。下面用一个 mock LLM 演示 ReAct 的控制流（不依赖真实模型）。


In [ ]:
import re


# ============== 简单工具实现 ==============
def calculator(expr):
    """安全的算术求值。"""
    try:
        # 只允许数字和算术符号
        if re.match(r'^[0-9+\-*/(). ]+$', expr):
            return str(eval(expr))
    except Exception as e:
        return f'error: {e}'
    return 'error: invalid expression'


def search(query):
    """Mock 搜索：从一个固定知识库里返回。"""
    kb = {
        'capital of france': 'Paris',
        'speed of light': '299792458 m/s',
        'pi': '3.14159265358979',
        'first president of usa': 'George Washington',
    }
    q = query.lower().strip()
    for k, v in kb.items():
        if k in q:
            return v
    return 'no result'


TOOLS = {
    'Calculator': calculator,
    'Search': search,
}


# ============== Mock LLM：根据问题硬编码 ReAct 轨迹 ==============
def mock_llm(prompt, history):
    """
    把问题 + 历史观察拼起来，输出下一步 Thought/Action/Answer。
    真实智能体里这一步是调用 GPT-4 / Claude 等。
    """
    text = prompt
    for step in history:
        text += f'\n{step}'

    # 简单规则：根据问题里的关键词决定动作
    if 'Answer:' in text and text.rstrip().endswith('Answer:'):
        return None  # 终止

    # 第一步：分析问题，决定调哪个工具
    if not history:
        if any(op in prompt for op in '+-*/×÷'):
            return 'Thought: 这是个算术题，我需要 Calculator。\nAction: Calculator("' + re.search(r'[\d+\-*/().× ÷]+', prompt).group().replace('×', '*').replace('÷', '/').strip() + '")'
        else:
            # 抽取问题主体
            q = prompt.replace('问', '').replace('?', '').replace('？', '').strip()
            return f'Thought: 我需要查询信息。\nAction: Search("{q}")'

    # 第二步：根据观察给出最终答案
    last = history[-1]
    if last.startswith('Observation:'):
        result = last.replace('Observation:', '').strip()
        return f'Thought: 我得到了答案。\nAnswer: {result}'

    return None


# ============== ReAct 主循环 ==============
def react_loop(question, max_steps=5, verbose=True):
    prompt = f'Question: {question}'
    history = []

    for step in range(max_steps):
        # 1. 让 LM 输出下一步
        out = mock_llm(prompt, history)
        if out is None:
            break
        if verbose:
            print(out)
        history.append(out)

        # 2. 检查是否要执行 Action
        m = re.search(r'Action: (\w+)\("(.+?)"\)', out)
        if m:
            tool_name, tool_input = m.group(1), m.group(2)
            if tool_name in TOOLS:
                obs = TOOLS[tool_name](tool_input)
            else:
                obs = f'error: unknown tool {tool_name}'
            obs_line = f'Observation: {obs}'
            if verbose: print(obs_line)
            history.append(obs_line)
            continue

        # 3. 检查是否要终止
        if 'Answer:' in out:
            break

    return history


# 演示
print('=' * 60)
print('Example 1: 算术题')
print('=' * 60)
react_loop('帮我算一下 (15 + 27) * 3')
print()
print('=' * 60)
print('Example 2: 查询事实')
print('=' * 60)
react_loop('What is the capital of France')


## 5. RAG：检索增强生成

智能体除了主动调工具，还有一种常见模式：**检索增强生成**（Retrieval-Augmented Generation，RAG）。

流程：

1. 把知识库切成小块（chunks），用 embedding 模型编码成向量；
2. 把向量存进向量数据库（FAISS、Milvus 等）；
3. 用户提问时，把问题编码成向量，**检索**最相似的几个 chunks；
4. 把这些 chunks 作为上下文塞到 LM 的 prompt 里，让 LM 基于检索到的信息回答。

最小化的 RAG 实现演示如下：


In [ ]:
# ============== Mock embedding：用字符 n-gram 做简单近似 ==============
def simple_embed(text, dim=64):
    """用字符 trigram 出现频率做一个超粗糙的 embedding。"""
    text = text.lower()
    vec = torch.zeros(dim)
    for i in range(len(text) - 2):
        tri = text[i:i+3]
        h = sum(ord(c) for c in tri) % dim
        vec[h] += 1
    # 单位化
    norm = vec.norm()
    return vec / (norm + 1e-9)


# ============== 小型知识库 ==============
kb_chunks = [
    'PyTorch 是 Meta 开源的深度学习框架，2016 年发布。它采用动态计算图。',
    '飞桨（PaddlePaddle）是百度开源的国产深度学习框架，2016 年发布。',
    'TensorFlow 是 Google 开源的深度学习框架，2015 年发布。早期版本使用静态图。',
    'JAX 是 Google 开源的用于科学计算的库，结合 NumPy 接口和自动微分。',
    'Transformer 由 Vaswani 等人在 2017 年提出，是当前 LLM 的核心架构。',
    'GPT 系列由 OpenAI 开发，包含 GPT-1（2018）、GPT-2（2019）、GPT-3（2020）、GPT-4（2023）。',
    'Llama 系列由 Meta 开源，包含 Llama 1（2023.2）、Llama 2（2023.7）、Llama 3（2024.4）。',
]

# 预先把所有 chunk embed
kb_embeddings = torch.stack([simple_embed(c) for c in kb_chunks])
print(f'knowledge base: {len(kb_chunks)} chunks')


# ============== RAG 主流程 ==============
def rag(question, k=2):
    # 1. 嵌入问题
    q_vec = simple_embed(question)
    # 2. 检索最相似的 k 个 chunk
    sims = kb_embeddings @ q_vec                           # 余弦相似度（已经单位化）
    topk = sims.topk(k).indices.tolist()
    retrieved = [kb_chunks[i] for i in topk]
    print(f'\nQuestion: {question}')
    print(f'Retrieved {k} most relevant chunks:')
    for i, c in zip(topk, retrieved):
        print(f'  [{i}] sim={sims[i]:.3f}: {c}')
    # 3. 把检索到的 chunk 拼到 prompt 里（mock 一下 LM 的回答）
    context = '\n'.join(retrieved)
    answer = f'(based on retrieved context) 这是根据检索到的 {len(retrieved)} 条相关知识合成的回答。'
    return answer


rag('PyTorch 是什么时候发布的？')
rag('GPT-4 是哪一年的？')
rag('Llama 2 何时开源？')


## 小结

本节演示了让 LLM 从"会说话"到"会做事"的几个关键能力：

| 技术 | 解决什么 | 关键代码 |
|---|---|---|
| **LoRA** | 高效微调，省显存 | `LoRALinear` 包装 + 冻结底层 |
| **SFT** | 让模型听指令 | 在 (prompt, response) 上训练，只在 response 段算 loss |
| **DPO** | 让模型偏好某种风格 | $-\log \sigma(\beta [\Delta_w - \Delta_l])$ 单一对比损失 |
| **ReAct** | 让模型主动调工具 | Thought / Action / Observation 循环 |
| **RAG** | 让模型查知识库 | 向量检索 + 检索结果拼 prompt |

**实战提示**：

- 真实生产中，**SFT** 用 [`trl.SFTTrainer`](https://huggingface.co/docs/trl) 或自己写循环；
- **DPO** 用 [`trl.DPOTrainer`](https://huggingface.co/docs/trl/dpo_trainer)；
- **智能体** 用 [LangChain](https://www.langchain.com/) / [LlamaIndex](https://www.llamaindex.ai/) / [AutoGen](https://microsoft.github.io/autogen/) 等框架；
- **RAG** 用 [LangChain](https://www.langchain.com/)、[LlamaIndex](https://www.llamaindex.ai/) 加 FAISS / Milvus / Chroma 等向量库。

本章只是把这些技术的**核心思想**剥离出来用 100-200 行代码演示——理解了原理，再去用工业级框架就驾轻就熟。

> **大语言模型还有很多没讲到的话题**：MoE 混合专家、思维链推理、强化学习推理（RLHF/GRPO）、长上下文、多模态、agent-as-a-judge 等。后续可以参考《神经网络与深度学习》理论书 v2 的第 13 章，或专门的 [大模型与智能体](https://github.com/nndl/llm-agent) 教材。
